In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import poisson

# New variable names for clarity
x_vals = np.sort(X)  # standardized PCTCOVER
theta_hat = beta_hat  # MLE parameters
lambda_curve = np.exp(theta_hat[0] + theta_hat[1] * x_vals)  # fitted λ values

# Prediction bounds
lower_bound = poisson.ppf(0.05, lambda_curve)
upper_bound = poisson.ppf(0.95, lambda_curve)

# Start plot
plt.scatter(X, y,
            label="Observed Salamanders",
            xlabel="Standardized Groundcover",
            ylabel="Count",
            title="Poisson Model Fit + 90% Bounds",
            color='seagreen', s=6, alpha=0.8)

# Add Poisson fit line
plt.plot(x_vals, lambda_curve,
         label="Fitted Mean",
         color='crimson', linewidth=3)

# Add dashed quantile lines
plt.plot(x_vals, lower_bound,
         label="Lower Bound (5%)", linestyle='--', color='gray', linewidth=2)

plt.plot(x_vals, upper_bound,
         label="Upper Bound (95%)", linestyle='--', color='gray', linewidth=2)

# Show final plot
plt.legend(loc='upper left')
plt.grid(alpha=0.3)
plt.show()

# === Load and standardize data ===
file_path = r"C:\CORNELL WORK\SP25\BEE 5850\HW 4\salamanders.csv"
df = pd.read_csv(file_path, delimiter=';')
df['x_std'] = (df['PCTCOVER'] - df['PCTCOVER'].mean()) / df['PCTCOVER'].std()
x = df['x_std'].values
y = df['SALAMAN'].values
n = len(y)

# Poisson log-likelihood
def poisson_loglik(theta, y, x):
    beta_0, beta_1 = theta
    lambda_ = np.exp(beta_0 + beta_1 * x)
    return -np.sum(poisson.logpmf(y, lambda_))

def fit_poisson(x, y):
    result = minimize(lambda theta: poisson_loglik(theta, y, x), 
                      x0=[-10.0, -50.0], bounds=[(-10.0, 10.0), (-50.0, 50.0)])
    return result.x

# Fit original model
theta_mle = fit_poisson(x, y)
beta_0_mle, beta_1_mle = theta_mle

# Bootstrap function
def salamander_bootstrap(x, y, nsamp):
    intercepts = []
    coefficients = []
    n = len(y)

    for i in range(nsamp):
        idx = np.random.randint(0, n, n)
        x_b, y_b = x[idx], y[idx]
        try:
            theta_b = fit_poisson(x_b, y_b)
            intercepts.append(theta_b[0])
            coefficients.append(theta_b[1])
        except:
            continue

    return np.array(intercepts), np.array(coefficients)

n_boot = 1000
np.random.seed(42)
boot_beta_0, boot_beta_1 = salamander_bootstrap(x, y, n_boot)

# Means & Biases
mean_beta_0 = np.mean(boot_beta_0)
mean_beta_1 = np.mean(boot_beta_1)
bias_beta_0 = beta_0_mle - mean_beta_0
bias_beta_1 = beta_1_mle - mean_beta_1

# 90% Confidence Intervals
# Reflected (basic bootstrap)
ci_beta_0_reflected = 2 * beta_0_mle - np.quantile(boot_beta_0, [0.95, 0.05])
ci_beta_1_reflected = 2 * beta_1_mle - np.quantile(boot_beta_1, [0.95, 0.05])

# Percentile (nonparametric bootstrap)
ci_beta_0_percentile = np.quantile(boot_beta_0, [0.05, 0.95])
ci_beta_1_percentile = np.quantile(boot_beta_1, [0.05, 0.95])

print("\n--- Bootstrap Summary ---")
print("MLE β₀: ", round(beta_0_mle, 4))
print("  Bias: ", round(bias_beta_0, 4))
print("  90% Confidence Interval (Reflected): (", round(ci_beta_0_reflected[0], 4), ", ", round(ci_beta_0_reflected[1], 4), ")")
print("  90% Confidence Interval (Percentile): (", round(ci_beta_0_percentile[0], 4), ", ", round(ci_beta_0_percentile[1], 4), ")")

NameError: name 'X' is not defined

In [ ]:
data_dir = os.path.join("data", "salamanders.csv")
salamander_data = pd.read_csv(data_dir, index_col='SITE')